# PartB_2

This notebook explores fine-tuning as a transfer learning method for the flower dataset using a pretrained MobileNetV2 model.

Structure:
1. Utilities and data loading
2. Transfer learning model setup
3. Fine-tuning variants and benchmark

## Optional Colab Setup

If you are using Google Colab and the dataset is stored in Google Drive, run the following cell first and make sure `data1.h5` is available.

In [ ]:
from google.colab import drive
drive.mount("/content/gdrive")
# !unzip "/content/gdrive/MyDrive/Deep_learning_2/flower.h5.zip" -d "/content/gdrive/MyDrive/Deep_learning_2/"
!ls /content/gdrive/MyDrive/Deep_learning_2

## Utilities

This section contains the imports, data loader, tf.data pipeline helpers, and the evaluation/visualization utilities reused across the notebook.

In [ ]:
import csv
import time
import h5py
import matplotlib.pyplot as plt
import numpy as np
import os
import tensorflow as tf
from keras import layers
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# source: https://www.tensorflow.org/tutorials/images/transfer_learning
# source: https://www.tensorflow.org/api_docs/python/tf/keras/applications/MobileNetV2
# source: https://www.tensorflow.org/api_docs/python/tf/keras/applications/mobilenet_v2/preprocess_input


In [ ]:
def loadDataH5():
    candidate_paths = [
        "data1.h5",
        "/content/data1.h5",
        "/content/gdrive/MyDrive/Deep_learning_2/data1.h5",
        "/content/drive/MyDrive/Deep_learning_2/data1.h5",
    ]

    data_path = None
    for candidate in candidate_paths:
        if os.path.exists(candidate):
            data_path = candidate
            break

    if data_path is None:
        raise FileNotFoundError(
            "Could not find data1.h5. Expected it in the current directory, "
            "/content, or /content/gdrive/MyDrive/Deep_learning_2."
        )

    print("Using data file:", data_path)

    with h5py.File(data_path, "r") as hf:
        trainX = np.array(hf.get("trainX"))
        trainY = np.array(hf.get("trainY"))
        valX = np.array(hf.get("valX"))
        valY = np.array(hf.get("valY"))

    print("trainX shape:", trainX.shape, "trainY shape:", trainY.shape)
    print("valX shape:", valX.shape, "valY shape:", valY.shape)
    return trainX, trainY, valX, valY


def build_tf_datasets(trainX, trainY, valX, valY, batch_size=32):
    train_ds = tf.data.Dataset.from_tensor_slices((trainX, trainY))
    train_ds = train_ds.shuffle(len(trainX)).batch(batch_size).prefetch(tf.data.AUTOTUNE)

    val_ds = tf.data.Dataset.from_tensor_slices((valX, valY))
    val_ds = val_ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

    return train_ds, val_ds


def plot_history(initial_history, fine_history, model_name, initial_epochs, output_dir="plots_partB2"):
    os.makedirs(output_dir, exist_ok=True)

    acc = initial_history.history["accuracy"] + fine_history.history["accuracy"]
    val_acc = initial_history.history["val_accuracy"] + fine_history.history["val_accuracy"]
    loss = initial_history.history["loss"] + fine_history.history["loss"]
    val_loss = initial_history.history["val_loss"] + fine_history.history["val_loss"]

    total_epochs = len(acc)
    epochs_range = range(total_epochs)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(epochs_range, acc, label="train_accuracy")
    axes[0].plot(epochs_range, val_acc, label="val_accuracy")
    axes[0].axvline(initial_epochs - 1, color="black", linestyle="--", label="start_fine_tuning")
    axes[0].set_title(f"{model_name} Accuracy")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Accuracy")
    axes[0].legend()

    axes[1].plot(epochs_range, loss, label="train_loss")
    axes[1].plot(epochs_range, val_loss, label="val_loss")
    axes[1].axvline(initial_epochs - 1, color="black", linestyle="--", label="start_fine_tuning")
    axes[1].set_title(f"{model_name} Loss")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"{model_name}_history.png"), dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def plot_confusion_matrix(y_true, y_pred, model_name, class_names, output_dir="plots_partB2"):
    os.makedirs(output_dir, exist_ok=True)
    cm = confusion_matrix(y_true, y_pred)
    row_sums = cm.sum(axis=1, keepdims=True)
    cm_normalized = np.divide(
        cm.astype("float"),
        row_sums,
        out=np.zeros_like(cm, dtype=float),
        where=row_sums != 0,
    )

    fig, ax = plt.subplots(figsize=(10, 8))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(ax=ax, cmap="Blues", xticks_rotation=45, colorbar=True)
    ax.set_title(f"{model_name} Confusion Matrix")
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"{model_name}_confusion_matrix.png"), dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(10, 8))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm_normalized, display_labels=class_names)
    disp.plot(ax=ax, cmap="Blues", xticks_rotation=45, colorbar=True, values_format=".2f")
    ax.set_title(f"{model_name} Normalized Confusion Matrix")
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"{model_name}_confusion_matrix_normalized.png"), dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    np.savetxt(
        os.path.join(output_dir, f"{model_name}_confusion_matrix.csv"),
        cm,
        delimiter=",",
        fmt="%d",
    )
    np.savetxt(
        os.path.join(output_dir, f"{model_name}_confusion_matrix_normalized.csv"),
        cm_normalized,
        delimiter=",",
        fmt="%.6f",
    )

    return cm, cm_normalized


def save_classification_report(y_true, y_pred, class_names, model_name, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    report_text = classification_report(y_true, y_pred, target_names=class_names)
    report_dict = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)

    txt_path = os.path.join(output_dir, f"{model_name}_classification_report.txt")
    with open(txt_path, "w") as report_file:
        report_file.write(report_text)

    csv_path = os.path.join(output_dir, f"{model_name}_classification_report.csv")
    with open(csv_path, "w", newline="") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["label", "precision", "recall", "f1-score", "support"])
        for label, metrics in report_dict.items():
            if isinstance(metrics, dict):
                writer.writerow([
                    label,
                    metrics.get("precision"),
                    metrics.get("recall"),
                    metrics.get("f1-score"),
                    metrics.get("support"),
                ])


def plot_prediction_examples(images, y_true, y_pred, true_class, pred_class=None, max_images=6, model_name="model", output_dir="plots_partB2"):
    os.makedirs(output_dir, exist_ok=True)

    if pred_class is None:
        indices = np.where((y_true == true_class) & (y_pred == true_class))[0]
        title = f"{model_name}: correctly classified class {true_class}"
        file_name = f"{model_name}_class_{true_class}_correct_examples.png"
    else:
        indices = np.where((y_true == true_class) & (y_pred == pred_class))[0]
        title = f"{model_name}: class {true_class} misclassified as class {pred_class}"
        file_name = f"{model_name}_class_{true_class}_pred_{pred_class}_examples.png"

    if len(indices) == 0:
        print(f"No matching examples found for {title}.")
        return

    indices = indices[:max_images]
    fig, axes = plt.subplots(1, len(indices), figsize=(3 * len(indices), 3))
    if len(indices) == 1:
        axes = [axes]

    for ax, idx in zip(axes, indices):
        ax.imshow(images[idx])
        ax.set_title(f"true={y_true[idx]}\npred={y_pred[idx]}")
        ax.axis("off")

    plt.suptitle(title)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, file_name), dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def get_top_confusion_pair(cm):
    confusion_only = cm.copy()
    np.fill_diagonal(confusion_only, 0)
    max_index = np.argmax(confusion_only)
    true_class, pred_class = np.unravel_index(max_index, confusion_only.shape)
    if confusion_only[true_class, pred_class] == 0:
        return None
    return true_class, pred_class


def plot_class_confusion_comparison(images, y_true, y_pred, true_class, pred_class, max_images=4, model_name="model", output_dir="plots_partB2"):
    os.makedirs(output_dir, exist_ok=True)
    misclassified_idx = np.where((y_true == true_class) & (y_pred == pred_class))[0]
    true_correct_idx = np.where((y_true == true_class) & (y_pred == true_class))[0]
    pred_correct_idx = np.where((y_true == pred_class) & (y_pred == pred_class))[0]

    if len(misclassified_idx) == 0:
        print(f"No misclassified examples found for class {true_class} predicted as class {pred_class}.")
        return

    misclassified_idx = misclassified_idx[:max_images]
    true_correct_idx = true_correct_idx[:max_images]
    pred_correct_idx = pred_correct_idx[:max_images]

    columns = max(len(misclassified_idx), len(true_correct_idx), len(pred_correct_idx), 1)
    fig, axes = plt.subplots(3, columns, figsize=(3 * columns, 9))
    if columns == 1:
        axes = np.array(axes).reshape(3, 1)

    row_titles = [
        f"Misclassified: true={true_class}, pred={pred_class}",
        f"Correct examples of true class {true_class}",
        f"Correct examples of predicted class {pred_class}",
    ]
    row_indices = [misclassified_idx, true_correct_idx, pred_correct_idx]

    for row, (title, indices) in enumerate(zip(row_titles, row_indices)):
        for col in range(columns):
            ax = axes[row, col]
            if col < len(indices):
                idx = indices[col]
                ax.imshow(images[idx])
                ax.set_title(f"true={y_true[idx]}\npred={y_pred[idx]}")
                ax.axis("off")
            else:
                ax.axis("off")
        axes[row, 0].set_ylabel(title, rotation=90, fontsize=11, labelpad=20)

    plt.suptitle(f"{model_name}: visual comparison for confusion {true_class} -> {pred_class}", fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"{model_name}_class_{true_class}_vs_class_{pred_class}_comparison.png"), dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)


## Model Setup

This section follows the TensorFlow fine-tuning tutorial logic:
1. Create a pretrained MobileNetV2 base model.
2. Freeze the base model and train the classification head first.
3. Unfreeze the top part of the base model.
4. Recompile with a lower learning rate.
5. Continue training.

In [ ]:
IMG_SIZE = (128, 128)
IMG_SHAPE = IMG_SIZE + (3,)
BATCH_SIZE = 32
INITIAL_EPOCHS = 10
FINE_TUNE_EPOCHS = 10
BASE_LEARNING_RATE = 1e-4

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
], name="data_augmentation")


def build_finetune_model(num_classes=17, dropout_rate=0.2, use_augmentation=False):
    base_model = tf.keras.applications.MobileNetV2(
        input_shape=IMG_SHAPE,
        include_top=False,
        weights="imagenet",
    )
    base_model.trainable = False

    inputs = tf.keras.Input(shape=IMG_SHAPE)
    x = inputs
    if use_augmentation:
        x = data_augmentation(x)
    x = tf.keras.applications.mobilenet_v2.preprocess_input(x * 255.0)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(dropout_rate)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = tf.keras.Model(inputs, outputs)
    return model, base_model


def compile_initial_model(model):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=BASE_LEARNING_RATE),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )


def apply_fine_tuning(base_model, fine_tune_at):
    base_model.trainable = True
    print("Number of layers in the base model:", len(base_model.layers))
    print("Fine-tune from this layer onwards:", fine_tune_at)

    for layer in base_model.layers[:fine_tune_at]:
        layer.trainable = False


def compile_finetuned_model(model, learning_rate):
    model.compile(
        optimizer=tf.keras.optimizers.RMSprop(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )


## Fine-Tuning Variants

This section defines three variants:
- Variant 1: fine-tune from layer 100 onward
- Variant 2: fine-tune from layer 120 onward
- Variant 3: fine-tune from layer 100 onward with augmentation and stronger dropout

In [ ]:
def evaluate_predictions(valX, valY, y_pred, model_name, output_dir="plots_partB2"):
    class_names = [f"Class {index}" for index in range(17)]
    cm, cm_normalized = plot_confusion_matrix(valY, y_pred, model_name, class_names, output_dir=output_dir)

    print(f"\nClassification report for {model_name}:")
    print(classification_report(valY, y_pred, target_names=class_names))
    save_classification_report(valY, y_pred, class_names, model_name, output_dir)

    best_class = int(np.argmax(np.diag(cm_normalized)))
    best_class_score = np.diag(cm_normalized)[best_class]
    print(f"Best classified class for {model_name}: class {best_class} with normalized recall {best_class_score:.2f}")

    plot_prediction_examples(valX, valY, y_pred, true_class=best_class, pred_class=None, max_images=5, model_name=model_name, output_dir=output_dir)

    top_confusion = get_top_confusion_pair(cm)
    if top_confusion is not None:
        true_class, pred_class = top_confusion
        plot_prediction_examples(valX, valY, y_pred, true_class=true_class, pred_class=pred_class, max_images=5, model_name=model_name, output_dir=output_dir)
        plot_class_confusion_comparison(valX, valY, y_pred, true_class=true_class, pred_class=pred_class, max_images=4, model_name=model_name, output_dir=output_dir)


def run_finetune_variant(train_ds, val_ds, valX, valY, variant_name, fine_tune_at, dropout_rate=0.2, use_augmentation=False):
    output_dir = "plots_partB2"
    os.makedirs(output_dir, exist_ok=True)

    model, base_model = build_finetune_model(
        num_classes=17,
        dropout_rate=dropout_rate,
        use_augmentation=use_augmentation,
    )

    compile_initial_model(model)

    initial_start = time.perf_counter()
    history_initial = model.fit(
        train_ds,
        epochs=INITIAL_EPOCHS,
        validation_data=val_ds,
        verbose=1,
    )
    initial_training_seconds = time.perf_counter() - initial_start

    apply_fine_tuning(base_model, fine_tune_at=fine_tune_at)
    compile_finetuned_model(model, learning_rate=BASE_LEARNING_RATE / 10)

    fine_start = time.perf_counter()
    history_fine = model.fit(
        train_ds,
        epochs=INITIAL_EPOCHS + FINE_TUNE_EPOCHS,
        initial_epoch=len(history_initial.epoch),
        validation_data=val_ds,
        verbose=1,
    )
    fine_tuning_seconds = time.perf_counter() - fine_start

    eval_start = time.perf_counter()
    val_loss, val_accuracy = model.evaluate(val_ds, verbose=0)
    predict_start = time.perf_counter()
    y_pred_probs = model.predict(val_ds, verbose=0)
    predict_seconds = time.perf_counter() - predict_start
    evaluation_seconds = time.perf_counter() - eval_start
    y_pred = np.argmax(y_pred_probs, axis=1)

    print(f"\n{variant_name} validation loss: {val_loss:.4f}")
    print(f"{variant_name} validation accuracy: {val_accuracy:.4f}")
    print(f"{variant_name} initial training time: {initial_training_seconds:.2f}s")
    print(f"{variant_name} fine-tuning time: {fine_tuning_seconds:.2f}s")
    print(f"{variant_name} prediction time: {predict_seconds:.4f}s ({(predict_seconds / len(valY)) * 1000:.3f} ms/image)")

    plot_history(history_initial, history_fine, variant_name, INITIAL_EPOCHS, output_dir=output_dir)
    evaluate_predictions(valX, valY, y_pred, variant_name, output_dir=output_dir)

    return {
        "val_loss": val_loss,
        "val_accuracy": val_accuracy,
        "history_initial": history_initial.history,
        "history_fine": history_fine.history,
        "initial_training_seconds": initial_training_seconds,
        "fine_tuning_seconds": fine_tuning_seconds,
        "total_training_seconds": initial_training_seconds + fine_tuning_seconds,
        "evaluation_seconds": evaluation_seconds,
        "predict_seconds": predict_seconds,
        "predict_ms_per_image": (predict_seconds / len(valY)) * 1000,
    }


def run_frozen_baseline_variant(train_ds, val_ds, valX, valY, variant_name="baseline_frozen_base", dropout_rate=0.2, use_augmentation=False):
    output_dir = "plots_partB2"
    os.makedirs(output_dir, exist_ok=True)

    model, _ = build_finetune_model(
        num_classes=17,
        dropout_rate=dropout_rate,
        use_augmentation=use_augmentation,
    )

    compile_initial_model(model)

    initial_start = time.perf_counter()
    history_initial = model.fit(
        train_ds,
        epochs=INITIAL_EPOCHS,
        validation_data=val_ds,
        verbose=1,
    )
    initial_training_seconds = time.perf_counter() - initial_start

    eval_start = time.perf_counter()
    val_loss, val_accuracy = model.evaluate(val_ds, verbose=0)
    predict_start = time.perf_counter()
    y_pred_probs = model.predict(val_ds, verbose=0)
    predict_seconds = time.perf_counter() - predict_start
    evaluation_seconds = time.perf_counter() - eval_start
    y_pred = np.argmax(y_pred_probs, axis=1)

    print(f"\n{variant_name} validation loss: {val_loss:.4f}")
    print(f"{variant_name} validation accuracy: {val_accuracy:.4f}")
    print(f"{variant_name} training time: {initial_training_seconds:.2f}s")
    print(f"{variant_name} prediction time: {predict_seconds:.4f}s ({(predict_seconds / len(valY)) * 1000:.3f} ms/image)")

    evaluate_predictions(valX, valY, y_pred, variant_name, output_dir=output_dir)

    return {
        "val_loss": val_loss,
        "val_accuracy": val_accuracy,
        "history_initial": history_initial.history,
        "history_fine": {},
        "initial_training_seconds": initial_training_seconds,
        "fine_tuning_seconds": 0.0,
        "total_training_seconds": initial_training_seconds,
        "evaluation_seconds": evaluation_seconds,
        "predict_seconds": predict_seconds,
        "predict_ms_per_image": (predict_seconds / len(valY)) * 1000,
    }


def plot_variant_comparison(results, output_dir="plots_partB2"):
    os.makedirs(output_dir, exist_ok=True)
    model_names = list(results.keys())
    accuracies = [results[name]["val_accuracy"] for name in model_names]
    colors = ["#8172B2", "#4C72B0", "#55A868", "#C44E52"][:len(model_names)]

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(model_names, accuracies, color=colors)
    ax.set_title("PartB_2 Validation Accuracy Comparison")
    ax.set_xlabel("Fine-Tuning Variant")
    ax.set_ylabel("Validation Accuracy")
    ax.set_ylim(0, 1)

    for index, accuracy in enumerate(accuracies):
        ax.text(index, accuracy + 0.01, f"{accuracy:.3f}", ha="center")

    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "partB2_variant_accuracy_comparison.png"), dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def run_all_variants(trainX, trainY, valX, valY):
    train_ds, val_ds = build_tf_datasets(trainX, trainY, valX, valY, batch_size=BATCH_SIZE)

    variant_configs = {
        "baseline_frozen_base": {"dropout_rate": 0.2, "use_augmentation": False, "is_baseline": True},
        "variant_1_finetune_at_100": {"fine_tune_at": 100, "dropout_rate": 0.2, "use_augmentation": False},
        "variant_2_finetune_at_120": {"fine_tune_at": 120, "dropout_rate": 0.2, "use_augmentation": False},
        "variant_3_augmented_dropout": {"fine_tune_at": 100, "dropout_rate": 0.3, "use_augmentation": True},
    }

    results = {}

    for variant_name, config in variant_configs.items():
        print(f"\n{'=' * 70}")
        print(f"Running {variant_name}")
        print(f"{'=' * 70}")

        if config.get("is_baseline"):
            results[variant_name] = run_frozen_baseline_variant(
                train_ds,
                val_ds,
                valX,
                valY,
                variant_name=variant_name,
                dropout_rate=config["dropout_rate"],
                use_augmentation=config["use_augmentation"],
            )
        else:
            results[variant_name] = run_finetune_variant(
                train_ds,
                val_ds,
                valX,
                valY,
                variant_name=variant_name,
                fine_tune_at=config["fine_tune_at"],
                dropout_rate=config["dropout_rate"],
                use_augmentation=config["use_augmentation"],
            )

    plot_variant_comparison(results)

    print("\nFinal results summary:")
    for variant_name, metrics in results.items():
        print(
            f"{variant_name}: val_loss={metrics['val_loss']:.4f}, "
            f"val_accuracy={metrics['val_accuracy']:.4f}, "
            f"train_time={metrics['total_training_seconds']:.2f}s, "
            f"predict_time={metrics['predict_seconds']:.4f}s"
        )

    best_variant = max(results, key=lambda name: results[name]["val_accuracy"])
    print(f"\nBest variant: {best_variant} with validation accuracy {results[best_variant]['val_accuracy']:.4f}")

    return results


## Run the Benchmark

Execute the following cell to load the flower dataset and compare the three fine-tuning variants.

In [ ]:
trainX, trainY, valX, valY = loadDataH5()
results = run_all_variants(trainX, trainY, valX, valY)

## Export Results to CSV

Export the main metrics to CSV after running the benchmark.

In [ ]:
import csv
import os
import shutil
import time


def export_partb2_results(results, output_dir="exports_partB2", copy_to_drive=True):
    os.makedirs(output_dir, exist_ok=True)

    summary_path = os.path.join(output_dir, "partB2_summary.csv")
    with open(summary_path, "w", newline="") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["variant_name", "val_loss", "val_accuracy", "initial_training_seconds", "fine_tuning_seconds", "total_training_seconds", "evaluation_seconds", "predict_seconds", "predict_ms_per_image"])
        for variant_name, metrics in results.items():
            writer.writerow([
                variant_name,
                metrics.get("val_loss"),
                metrics.get("val_accuracy"),
                metrics.get("initial_training_seconds"),
                metrics.get("fine_tuning_seconds"),
                metrics.get("total_training_seconds"),
                metrics.get("evaluation_seconds"),
                metrics.get("predict_seconds"),
                metrics.get("predict_ms_per_image"),
            ])

    history_path = os.path.join(output_dir, "partB2_history.csv")
    with open(history_path, "w", newline="") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["variant_name", "stage", "epoch", "accuracy", "val_accuracy", "loss", "val_loss"])
        for variant_name, metrics in results.items():
            for stage_name in ["history_initial", "history_fine"]:
                history = metrics.get(stage_name, {})
                epochs = len(history.get("accuracy", []))
                for epoch in range(epochs):
                    writer.writerow([
                        variant_name,
                        stage_name,
                        epoch + 1,
                        history.get("accuracy", [None] * epochs)[epoch],
                        history.get("val_accuracy", [None] * epochs)[epoch],
                        history.get("loss", [None] * epochs)[epoch],
                        history.get("val_loss", [None] * epochs)[epoch],
                    ])

    master_path = os.path.join(output_dir, "partB2_master_summary.csv")
    with open(master_path, "w", newline="") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["question", "model_name", "val_loss", "val_accuracy", "training_seconds", "predict_seconds", "predict_ms_per_image", "short_comment"])
        for variant_name, metrics in results.items():
            writer.writerow(["PartB_2", variant_name, metrics.get("val_loss"), metrics.get("val_accuracy"), metrics.get("total_training_seconds"), metrics.get("predict_seconds"), metrics.get("predict_ms_per_image"), ""])

    print(f"Saved CSV files to {output_dir}")
    print(summary_path)
    print(history_path)

    if copy_to_drive:
        drive_targets = [
            "/content/gdrive/MyDrive/Deep_learning_2",
            "/content/drive/MyDrive/Deep_learning_2",
        ]
        copied = False
        for drive_dir in drive_targets:
            if os.path.isdir(drive_dir):
                drive_export_dir = os.path.join(drive_dir, output_dir)
                os.makedirs(drive_export_dir, exist_ok=True)
                shutil.copy2(summary_path, os.path.join(drive_export_dir, os.path.basename(summary_path)))
                shutil.copy2(history_path, os.path.join(drive_export_dir, os.path.basename(history_path)))
                print(f"Copied CSV files to {drive_export_dir}")
                plot_source_dir = "plots_partB2"
                if os.path.isdir(plot_source_dir):
                    drive_plot_dir = os.path.join(drive_dir, plot_source_dir)
                    os.makedirs(drive_plot_dir, exist_ok=True)
                    for file_name in os.listdir(plot_source_dir):
                        source_file = os.path.join(plot_source_dir, file_name)
                        if os.path.isfile(source_file):
                            shutil.copy2(source_file, os.path.join(drive_plot_dir, file_name))
                    print(f"Copied plot files to {drive_plot_dir}")
                copied = True
                break
        if not copied:
            print("Google Drive export folder not found. CSV files were saved locally only.")


if "results" in globals():
    export_partb2_results(results)
else:
    print("Run the benchmark cell first, then rerun this export cell.")
